## 1. Imports

First we have to import plotting and loading functions designed for this P&S. In a next cell, you can add the imports you need. 
Please do not modify the original functions. If you think there is an error within, inform us. 

In [ ]:
from utils.data import load_data, save_data
from utils.plotting import create_post_stim_raster_plot, map_colour_to_electrode

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Settings

Here you can load a specific recording for either a shared or your network. A list of all existing configurations you can find on moodle. 

In [ ]:
Network = 5
DIV     = 40
group_data  = False

## 3. Data Loading and visualization

You will get various datapoints from a recording:

stimulation_parameters are the parameter used for each stimulus. The matrix shape is (n_stimulus, n_parameters). 
Currently there is frequency + pattern. Frequencies are floats, and pattern is an integer that encodes what electrodes where stimulated. 

stimulation_patterns contains solely the pattern applied for each stimulus. The matrix shape is (n_stimulus, 1). 

binned_spike_train_responses are the responses encoded in a binary spike train. For more efficient data handling, spikes are grouped in bins. The tensor is of shape (n_stimulus, n_recording_electrodes, n_time_bins). The current size of a time bin is 250µs. 

impedance_map is a 2D matrix that resembles where the microstructure lies on the CMOS recording area. 

electrodes are the electrode numbers of each recording electrode. They are sorted identically to the recording electrode dimension in binned_spike_train_responses. 

In [ ]:
data = load_data(Network, DIV, group_data)
stimulation_parameters, stimulation_patterns, binned_spike_train_responses, stimulation_times, impedance_map, electrodes = data

In [ ]:
# Here we color the recording electrodes
fig, ax = plt.subplots()
map_colour_to_electrode(ax, impedance_map, electrodes)
plt.show()

In [ ]:
pattern = 4
frequency = 30
indices = np.logical_and(stimulation_parameters[:,0]==frequency,stimulation_parameters[:,1]==pattern)

image = create_post_stim_raster_plot(binned_spike_train_responses[indices],electrodes)

plt.imshow(image, aspect='auto')
plt.gca().invert_yaxis()
plt.xlabel("Time Bin")
plt.ylabel("Stimulation")
plt.tight_layout()
plt.show()

In [ ]:
# X shape: n_stimulus x n_recording_electrodes x n_time_bins
X = binned_spike_train_responses

mean_response_over_time = X.mean(axis=(0, 1))

plt.figure(figsize=(7, 3))
plt.plot(mean_response_over_time)
plt.xlabel("Time bin")
plt.ylabel("Mean spike probability")
plt.title("Average population response over time")
plt.tight_layout()
plt.show()

In [ ]:
# Select one condition to visualize
pattern = 4
frequency = 30

condition_indices = np.logical_and(
    stimulation_parameters[:, 0] == frequency,
    stimulation_parameters[:, 1] == pattern
)

X_condition = binned_spike_train_responses[condition_indices]

print("Number of trials for selected condition:", X_condition.shape[0])

# Average response across trials for this condition: electrodes x time
condition_mean = X_condition.mean(axis=0)

# Plot selected time bins as electrode maps
time_bins_to_plot = [0, 5, 10, 20, 40, 60]

fig, axes = plt.subplots(1, len(time_bins_to_plot), figsize=(3 * len(time_bins_to_plot), 3))

for ax, t in zip(axes, time_bins_to_plot):
    # Create an empty MEA-sized image and fill electrode locations.
    # This uses map_colour_to_electrode to show electrode positions first.
    map_colour_to_electrode(ax, impedance_map, electrodes)
    ax.set_title(f"t = {t}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("Recording electrode positions on MEA grid")
plt.tight_layout()
plt.show()

print("Note: the provided plotting utility shows electrode positions. To color each electrode by activity,")
print("you need the electrode-to-coordinate mapping from the data utilities or metadata.")


In [ ]:
# Plot each recording electrode by activity
# Activity = mean response across trials and time bins for each electrode

X = binned_spike_train_responses

# Handle either shape:
#   n_stimulus x n_electrodes x n_time_bins
# or
#   n_stimulus x 1 x n_electrodes x n_time_bins
if X.ndim == 4:
    X = X[:, 0, :, :]

# Optional: restrict to one condition
use_condition = False
frequency = 30
pattern = 4

if use_condition:
    indices = np.logical_and(
        stimulation_parameters[:, 0] == frequency,
        stimulation_parameters[:, 1] == pattern
    )
    X_plot = X[indices]
    title_suffix = f"frequency={frequency}, pattern={pattern}"
else:
    X_plot = X
    title_suffix = "all stimuli"

# Mean activity per electrode
electrode_activity = X_plot.mean(axis=(0, 2))

# Electrode labels
electrode_labels = np.asarray(electrodes)

# Sort electrodes from most active to least active
sort_idx = np.argsort(electrode_activity)[::-1]

plt.figure(figsize=(12, 4))
plt.bar(np.arange(len(electrode_activity)), electrode_activity[sort_idx])
plt.xticks(
    np.arange(len(electrode_activity)),
    electrode_labels[sort_idx],
    rotation=90,
    fontsize=7
)
plt.xlabel("Recording electrode")
plt.ylabel("Mean activity")
plt.title(f"Electrode activity ranked by response: {title_suffix}")
plt.tight_layout()
plt.show()

# Also show unsorted activity in original electrode order
plt.figure(figsize=(12, 4))
plt.plot(electrode_activity, marker="o", linewidth=1)
plt.xticks(
    np.arange(len(electrode_activity)),
    electrode_labels,
    rotation=90,
    fontsize=7
)
plt.xlabel("Recording electrode")
plt.ylabel("Mean activity")
plt.title(f"Electrode activity in original electrode order: {title_suffix}")
plt.tight_layout()
plt.show()

print("Most active electrodes:")
for rank, idx in enumerate(sort_idx[:10], start=1):
    print(f"{rank:2d}. Electrode {electrode_labels[idx]} | activity = {electrode_activity[idx]:.4f}")

In [ ]:
# Electrode position plot where color = activity
# This reuses map_colour_to_electrode for the positions,
# then replaces the colors with activity values.

import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1) Choose data to average
# -----------------------------
X = binned_spike_train_responses

# Handle either:
#   (n_stimulus, n_electrodes, n_time_bins)
# or
#   (n_stimulus, 1, n_electrodes, n_time_bins)
if X.ndim == 4:
    X = X[:, 0, :, :]

use_condition = False
frequency = 30
pattern = 4

if use_condition:
    mask = np.logical_and(
        stimulation_parameters[:, 0] == frequency,
        stimulation_parameters[:, 1] == pattern
    )
    X_plot = X[mask]
    title_suffix = f"freq={frequency} Hz, pattern={pattern}"
else:
    X_plot = X
    title_suffix = "all stimuli"

# Mean activity per electrode
electrode_activity = X_plot.mean(axis=(0, 2))

# -----------------------------
# 2) Use the existing plotting helper for electrode positions
# -----------------------------
fig, ax = plt.subplots(figsize=(7, 6))

# This already knows how to put the electrodes in the right place
map_colour_to_electrode(ax, impedance_map, electrodes)

# -----------------------------
# 3) Replace plotted colors with activity values
# -----------------------------
# Usually the electrode markers are stored in ax.collections.
# We find the collection with the same number of points as electrodes.
target_collection = None

for collection in ax.collections:
    try:
        offsets = collection.get_offsets()
        if len(offsets) == len(electrodes):
            target_collection = collection
            break
    except Exception:
        pass

if target_collection is None:
    raise RuntimeError(
        "Could not find the electrode scatter points created by map_colour_to_electrode. "
        "Run the debug cell below to inspect ax.collections."
    )

target_collection.set_array(np.asarray(electrode_activity))
target_collection.set_cmap("viridis")
target_collection.set_clim(
    np.nanmin(electrode_activity),
    np.nanmax(electrode_activity)
)

cbar = fig.colorbar(target_collection, ax=ax)
cbar.set_label("Mean activity")

ax.set_title(f"Electrode activity map ({title_suffix})")
plt.tight_layout()
plt.show()

# -----------------------------
# 4) Print top electrodes
# -----------------------------
sort_idx = np.argsort(electrode_activity)[::-1]

print("Top 10 most active electrodes:")
for rank, idx in enumerate(sort_idx[:10], start=1):
    print(f"{rank:2d}. Electrode {electrodes[idx]} | activity = {electrode_activity[idx]:.4f}")

In [ ]:
# Check whether CNN electrode order is spatially meaningful
# Color = electrode index in the CNN input
# If nearby colors are smooth/gradual, the CNN electrode axis preserves spatial structure.
# If colors look random, the CNN is not seeing true spatial adjacency.

import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))

# Use existing helper to plot electrode positions
map_colour_to_electrode(ax, impedance_map, electrodes)

# Find electrode scatter collection
target_collection = None
for collection in ax.collections:
    try:
        offsets = collection.get_offsets()
        if len(offsets) == len(electrodes):
            target_collection = collection
            break
    except Exception:
        pass

if target_collection is None:
    raise RuntimeError("Could not find electrode points from map_colour_to_electrode.")

# Color each electrode by its order in the CNN input
cnn_order = np.arange(len(electrodes))

target_collection.set_array(cnn_order)
target_collection.set_cmap("viridis")
target_collection.set_clim(cnn_order.min(), cnn_order.max())

cbar = fig.colorbar(target_collection, ax=ax)
cbar.set_label("Electrode index used by CNN")

ax.set_title("Does CNN electrode order match physical layout?")
plt.tight_layout()
plt.show()

In [ ]:
# Create a shuffled-electrode version of the response tensor
rng = np.random.default_rng(seed=42)
electrode_permutation = rng.permutation(X.shape[1])

X_shuffled_electrodes = X[:, electrode_permutation, :]

print("Original X shape:", X.shape)
print("Shuffled X shape:", X_shuffled_electrodes.shape)
print("First 10 shuffled electrode indices:", electrode_permutation[:10])

# Use X_shuffled_electrodes in the same training pipeline as the original CNN.
# Compare validation accuracy / balanced accuracy:
# - CNN(original electrode order)
# - CNN(shuffled electrode order)
#
# If original >> shuffled, spatial electrode order matters.
# If original ≈ shuffled, electrode-axis locality is probably not important.


In [ ]:
# Example feature extraction for simple baselines

# Spike count per electrode: n_stimulus x n_electrodes
spike_count_features = X.sum(axis=2)

# Total response over time: n_stimulus x n_time_bins
population_time_features = X.sum(axis=1)

# Flattened full response: n_stimulus x (n_electrodes * n_time_bins)
flattened_features = X.reshape(X.shape[0], -1)

# Labels for Task 1
y = stimulation_patterns.reshape(-1)

print("Spike-count features:", spike_count_features.shape)
print("Population-time features:", population_time_features.shape)
print("Flattened features:", flattened_features.shape)
print("Labels:", y.shape)